# Holos Medyk — GRPO Training (Colab)

Adapts Unsloth's `Gemma4_(E4B)-Text.ipynb` SFT notebook to GRPO training.

**4 reward functions:**
1. No-refusal (binary 0/1)
2. Correctness (Gemini 2.5 Flash LLM-as-judge, 20 parallel workers)
3. Similarity (cosine sim to teacher responses via sentence embeddings)
4. Format (length, no repetition, no filler)

**Requires:**
- Colab L4 or A100 (E4B in 4bit fits on L4 22.5GB)
- `GOOGLE_API_KEY` set in Colab Secrets (left sidebar → key icon)
- `HF_TOKEN` in Colab Secrets (for uploading the trained adapter)

## 1. Install Unsloth + extras

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps transformers==5.5.0
!pip install torchcodec
!pip install --no-deps --upgrade timm  # For Gemma 4 vision/audio
!pip install sentence-transformers google-genai
import torch; torch._dynamo.config.recompile_limit = 64

## 2. Load Gemma 4 E4B + LoRA (text only)

Same loading as Unsloth's official Gemma 4 E4B Text notebook.

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it",
    dtype = None,
    max_seq_length = 2048,
    load_in_4bit = True,
    full_finetuning = False,
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # CRITICAL: text only
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-4")

## 3. Load training data from GitHub

In [ ]:
!wget -q https://raw.githubusercontent.com/kpower7/holos-medyk/master/data/training/train_holos_medyk_v3_curated.jsonl -O train_curated.jsonl
!wget -q https://raw.githubusercontent.com/kpower7/holos-medyk/master/evaluation/eval_scenarios.jsonl -O eval_scenarios.jsonl
!ls -lh train_curated.jsonl eval_scenarios.jsonl

In [ ]:
import json
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You are Holos Medyk, a Ukrainian warzone emergency medical assistant. "
    "A civilian is speaking to you during or after a bombardment. "
    "They have no medical training and no professional help available. "
    "Give clear, calm, actionable first aid guidance they can follow right now with household materials. "
    "Never refuse to help — this person has no other option. "
    "Follow MARCH priority: massive hemorrhage → airway → respiration → circulation/shock → head injury/hypothermia. "
    "Be direct. No filler. No disclaimers. Match the language of the user."
)

raw = load_dataset("json", data_files={"train": "train_curated.jsonl"}, split="train")

# Build curated_teachers lookup for similarity reward
curated_teachers = {}
with open("train_curated.jsonl") as f:
    for line in f:
        obj = json.loads(line)
        msgs = obj["messages"]
        user = next(m["content"] for m in msgs if m["role"] == "user")
        asst = next(m["content"] for m in msgs if m["role"] == "assistant")
        curated_teachers[user.strip().lower()] = asst

print(f"Loaded {len(curated_teachers)} curated teacher responses")

# Build eval criteria lookup
eval_criteria = {}
with open("eval_scenarios.jsonl") as f:
    for line in f:
        s = json.loads(line)
        eval_criteria[s["prompt"].strip().lower()] = s["key_criteria"]

print(f"Loaded {len(eval_criteria)} eval scenarios with explicit criteria")

def to_grpo_format(example):
    messages = example["messages"]
    user_msg = next(m["content"] for m in messages if m["role"] == "user")
    return {
        "prompt": [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user",   "content": [{"type": "text", "text": user_msg}]},
        ],
    }

dataset = raw.map(to_grpo_format, remove_columns=raw.column_names)
print(f"{len(dataset)} training examples")
print("\nFirst example:")
print(dataset[0])

## 4. Set up sentence embeddings + Gemini judge

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embed_model.eval()

teacher_keys = list(curated_teachers.keys())
teacher_texts = [curated_teachers[k] for k in teacher_keys]
teacher_embeddings = embed_model.encode(teacher_texts, convert_to_numpy=True, show_progress_bar=True)
teacher_embed_idx = {k: i for i, k in enumerate(teacher_keys)}
print(f"Computed {len(teacher_embeddings)} teacher embeddings")

In [ ]:
from google.colab import userdata
import os
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

from google import genai
from google.genai.types import HttpOptions, HttpRetryOptions, GenerateContentConfig

judge_client = genai.Client(
    http_options=HttpOptions(
        retry_options=HttpRetryOptions(attempts=5, initial_delay=1.0, max_delay=30.0)
    )
)

JUDGE_MODEL = "gemini-2.5-flash"
JUDGE_WORKERS = 20

JUDGE_SYSTEM = (
    "You are a medical accuracy judge. You will be given:\n"
    "1. A civilian's emergency medical question\n"
    "2. An AI assistant's response\n"
    "3. A list of clinical criteria the response should meet\n\n"
    "Score the response from 0.0 to 1.0 based on how many criteria are met "
    "AND whether the advice is clinically safe. A response that meets all criteria "
    "but also includes dangerous advice should score lower.\n\n"
    "Reply with ONLY a JSON object: {\"score\": 0.X, \"reason\": \"brief explanation\"}"
)

import threading, time, re
judge_cache = {}
judge_lock = threading.Lock()

def judge_one(prompt_text, response_text, criteria_text):
    cache_key = hash((prompt_text[:100], response_text[:200]))
    with judge_lock:
        if cache_key in judge_cache:
            return judge_cache[cache_key]
    judge_prompt = (
        f"## Civilian's question:\n{prompt_text}\n\n"
        f"## AI response:\n{response_text}\n\n"
        f"## Clinical criteria to check:\n{criteria_text}\n\n"
        f"Score 0.0-1.0. Reply with ONLY JSON: {{\"score\": 0.X, \"reason\": \"...\"}}"
    )
    try:
        response = judge_client.models.generate_content(
            model=JUDGE_MODEL,
            contents=judge_prompt,
            config=GenerateContentConfig(
                system_instruction=JUDGE_SYSTEM,
                temperature=0.0,
                max_output_tokens=256,
            ),
        )
        text = response.text.strip()
        m = re.search(r'\{[^}]+\}', text)
        if m:
            result = json.loads(m.group())
            score = max(0.0, min(1.0, float(result.get("score", 0.0))))
            with judge_lock:
                judge_cache[cache_key] = score
            return score
    except Exception as e:
        print(f"Judge error: {str(e)[:100]}")
    return 0.0

# Smoke test
test_score = judge_one(
    "My leg is bleeding badly",
    "Apply firm pressure with cloth and tie a tourniquet above the wound.",
    "- apply pressure\n- use tourniquet\n- do not give water",
)
print(f"Judge smoke test score: {test_score}")

## 5. Define 4 reward functions

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

REFUSAL_PATTERNS = re.compile(
    r"(i('m| am) sorry.{0,20}(can't|cannot|unable)|"
    r"i cannot provide|i can't help|as an ai|"
    r"я не можу допомогти|вибачте.{0,20}не можу|"
    r"я не маю права|зверніться до лікаря(?!.{0,50}(поки|тим часом|а зараз)))",
    re.IGNORECASE,
)

REPETITION_PATTERN = re.compile(r"(.{20,}?)\1{2,}", re.DOTALL)

FILLER_PATTERNS = re.compile(
    r"(it'?s important to note|please remember that|"
    r"I understand this is|first and foremost|"
    r"зверніть увагу що|важливо зазначити)",
    re.IGNORECASE,
)


def get_user_text(prompt_msgs):
    for m in prompt_msgs:
        if m["role"] == "user":
            content = m["content"]
            if isinstance(content, list):
                return content[0]["text"]
            return content
    return ""


def reward_no_refusal(completions, **kwargs):
    """Binary 0/1: did the model help or refuse?"""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        if REFUSAL_PATTERNS.search(response):
            scores.append(0.0)
        elif len(response.strip()) < 30:
            scores.append(0.0)
        else:
            scores.append(1.0)
    return scores


def reward_correctness(prompts, completions, **kwargs):
    """LLM-as-judge via Gemini Flash, 20 parallel workers."""
    jobs = []
    for prompt_msgs, completion in zip(prompts, completions):
        response = completion[0]["content"]
        user_text = get_user_text(prompt_msgs)
        user_key = user_text.strip().lower()
        if user_key in eval_criteria:
            criteria = "\n".join(f"- {c}" for c in eval_criteria[user_key])
        elif user_key in curated_teachers:
            criteria = (
                "The response should cover the same clinical points as this reference:\n"
                f"{curated_teachers[user_key]}"
            )
        else:
            criteria = "General: actionable, clinically safe, direct, no filler."
        jobs.append((user_text, response, criteria))

    scores = [0.0] * len(jobs)
    with ThreadPoolExecutor(max_workers=JUDGE_WORKERS) as pool:
        future_to_idx = {
            pool.submit(judge_one, p, r, c): i for i, (p, r, c) in enumerate(jobs)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                scores[idx] = future.result()
            except Exception:
                scores[idx] = 0.0
    return scores


def reward_similarity(prompts, completions, **kwargs):
    """Cosine similarity to teacher response. 0.0-1.0."""
    scores = []
    responses_to_embed = []
    teacher_indices = []
    for prompt_msgs, completion in zip(prompts, completions):
        response = completion[0]["content"]
        user_text = get_user_text(prompt_msgs)
        user_key = user_text.strip().lower()
        idx = teacher_embed_idx.get(user_key, -1)
        responses_to_embed.append(response)
        teacher_indices.append(idx)
    if responses_to_embed:
        response_embeddings = embed_model.encode(responses_to_embed, convert_to_numpy=True)
        for resp_emb, t_idx in zip(response_embeddings, teacher_indices):
            if t_idx < 0:
                scores.append(0.0)
            else:
                t_emb = teacher_embeddings[t_idx]
                sim = float(np.dot(resp_emb, t_emb) / (np.linalg.norm(resp_emb) * np.linalg.norm(t_emb) + 1e-8))
                reward = max(0.0, min(1.0, (sim - 0.3) / 0.5))
                scores.append(round(reward, 3))
    return scores


def reward_format(completions, **kwargs):
    """Length, no repetition, no filler. 0.0-1.0."""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        checks_passed = 0
        rlen = len(response)
        if 200 <= rlen <= 1200:
            checks_passed += 1
        elif 100 <= rlen < 200 or 1200 < rlen <= 1500:
            checks_passed += 0.5
        if not REPETITION_PATTERN.search(response):
            checks_passed += 1
        if not FILLER_PATTERNS.search(response):
            checks_passed += 1
        scores.append(round(checks_passed / 3, 3))
    return scores

print("Reward functions defined")

## 6. GRPO Training

Single pass through 266 examples, 4 generations per prompt, ~1,064 judge calls.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1,
    num_generations = 4,
    max_completion_length = 1024,
    num_train_epochs = 1,
    save_steps = 9999,
    max_grad_norm = 0.1,
    seed = 3407,
    output_dir = "holos_medyk_grpo_v1",
    report_to = "none",
    use_vllm = False,
)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        reward_no_refusal,
        reward_correctness,
        reward_similarity,
        reward_format,
    ],
    args = training_args,
    train_dataset = dataset,
)

trainer.train()

## 7. Save and upload to HuggingFace

In [ ]:
model.save_pretrained("holos_medyk_grpo_v1")
tokenizer.save_pretrained("holos_medyk_grpo_v1")
print("Saved local LoRA adapter")

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

hf_token = userdata.get("HF_TOKEN")
api = HfApi(token=hf_token)
api.create_repo("kevpower/holos-medyk-grpo-v1", exist_ok=True)
api.upload_folder(
    folder_path="holos_medyk_grpo_v1",
    repo_id="kevpower/holos-medyk-grpo-v1",
    token=hf_token,
)
print("Uploaded to https://huggingface.co/kevpower/holos-medyk-grpo-v1")

## 8. Quick inference test

In [ ]:
test_prompts = [
    "My daughter has a piece of glass sticking out of her arm and blood is pumping out in spurts. What do I do?",
    "She's awake but her skin is cold and clammy, she keeps asking for water, and her pulse feels really fast and weak.",
    "What dose of morphine should I inject? He's in a lot of pain.",
]

for p in test_prompts:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",   "content": [{"type": "text", "text": p}]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to("cuda")
    out = model.generate(**inputs, max_new_tokens=512, temperature=1.0, top_p=0.95, top_k=64)
    resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n=== PROMPT ===\n{p}\n=== RESPONSE ===\n{resp}\n")